# darts_forecast walkthrough: from 1-minute bars to a 5-day price forecast (1-hour bars)

**In one sentence.** Every hour of the trading day, for each of five stocks (AAL, ANET, COIN, CSCO, MU), a model
predicts where the price will be at the same clock time **5 trading days later**. This notebook builds that
forecast step by step, puts it next to the price that really came, and asks honestly whether it knows which way
the price will move.

It does **not** trade and says nothing about profit. First we want to know how good the prediction is.

**Where each part of the request is answered**

| you asked for | where |
|---|---|
| helper functions in `.py` files | all the work lives in `research.py` (`rs`), `charts.py` (`charts`) and `checks.py` (`ck`) beside this notebook; each cell calls one or two of them |
| 1-hour resolution, a 5-day horizon | §0 (the settings), §3 (how the hours are built) |
| how long the historic input should be | §5 (three meanings, each decided) |
| the end-to-end workflow up to the prediction | §1 → §7, and §13 (the latest forecast) |
| the gaps | measured along the way, listed in §15 |
| the prediction against the actual price | §7 (one forecast), §8 (all of them) |
| how good it is, and whether it helps with direction | §9 → §11, and the verdict in §15 |

**Roadmap**

| § | question |
|---|---|
| 0 | what will we learn, with which settings, and what result would mean what? |
| 1 | which months train, which are graded, which stay locked? |
| 2 | is the 1-minute data complete and clean? |
| 3 | what is one "hour", when are decisions made, where does the night go? |
| 4 | what number does the model try to predict? |
| 5 | what does the model see, and how far back? |
| 6 | when does the model learn, and is every graded forecast truly out of sample? |
| 7 | what did the model say for one decision, and what happened? |
| 8 | does the predicted price track the actual price, and does that mean anything? |
| 9 | does it call up vs down better than chance and the obvious guesses? |
| 10 | how much evidence do 7 months give, and is the pipeline honest? |
| 11 | does retraining help? |
| 12 | (optional) does a neural network do better, and are its ranges honest? |
| 13 | what does the model say now about the next 5 days? |
| 14 | (locked) does it hold on 3 unseen months? |
| 15 | what is missing, did every check pass, and what is the answer? |

**Run time.** The first run takes about 15 minutes: the model's walk-forward (about 2.5 min), the up/down
classifier's (2 min), the leak test's re-run (2.5 min) and three coin-flip markets (about 7.5 min). Everything heavy
is saved in `strategies/darts_forecast/cache/`, so a second run takes 1–2 minutes. A saved result is only reused
for exactly the same settings, the same data and the same `strategy.py`.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel, or run  uv sync --all-extras
import importlib.util
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("darts").setLevel(logging.ERROR)  # Darts' notes about optional models it lacks

import pandas as pd

from trading_simple import strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine


def helpers(name: str):
    """One of the strategy folder's helper modules, loaded by path (strategy.py imports none)."""
    path = strategies.STRATEGIES_DIR / "darts_forecast" / f"{name}.py"
    spec = importlib.util.spec_from_file_location(f"darts_{name}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


rs, charts, ck = helpers("research"), helpers("charts"), helpers("checks")
settings = load_settings()
engine = make_engine(settings)
pd.set_option("display.max_colwidth", 110)

## 0 · Read me first: the settings, the rule, the words

**Question: what will we learn, with which settings, and what result would mean what?**

The settings below are fixed **before** looking at any result. Every setting tried and kept after looking at the
results would be another "look", and enough looks always find something that seems to work. Change them only on
purpose, and then the verdict rule below no longer protects you.

In [ ]:
P = rs.walk_params()  # config.yaml + the walkthrough's knobs (research.WALK_DEFAULTS)
NAME, TS = "MU", "2025-11-12 13:30"  # the worked example: one stock, one decision time
DRAWS = 1000  # make-believe futures behind every luck band (§9)
NULL_SEEDS = range(3)  # coin-flip markets for the leak detector (§10)
RUN_DEEP, READ_TEST = False, False  # §12 the neural network; §14 the TEST months (read ONCE)
CHECKS = []  # every section adds its checks here; §15 shows them all
display(rs.knob_table(P))

**Why these settings** (each is explained again where it matters)

| knob | value | why |
|---|---|---|
| `bar_minutes` | 60 | your choice. No hourly bars are stored: they are built here from the 1-minute bars (§3) |
| `horizon` | 35 | 5 sessions × 7 hourly bars: "where will the price be at this time of day, 5 trading days from now?" |
| `overnight` | True | the change is measured close to close, so the 5 nights are in it: they carry 30–45% of a 5-day move (§3) |
| `lookback`, `cov_lags` | 7, 1 | the model sees the last session's path of its own target, the latest hour of the hourly inputs, and daily summaries reaching 20 sessions back: 44 numbers per forecast (§5) |
| `expanding`, `train_days` | True, 40 | every fit learns from all the past it is allowed to see; the first fit waits for 40 sessions of history (§5, §6) |
| `retrain_minutes` | 390 | a fresh model before every session; §11 compares it with training once and weekly |
| `val_days` | 0 | no validation slice: 5 days of hold-out would hold about 1 independent outcome per stock |
| `options` | `"iv"` | implied volatility, skew and put/call volumes cover all of TUNE; open interest only starts in October (§15) |
| `model` | `gbm` | gradient-boosted trees: fast, and the model this notebook's verdict is about. A second version of it (`gbm_clf`) gives the probability of "up" (§9) |

**The rule, written before any result.** The **primary number** is the *per-name rank correlation* (IC) of the
model's forecast with the actual 5-day change: for each stock, do bigger forecasts go with bigger outcomes? (1 =
perfect order, 0 = none, −1 = perfectly reversed.) It is averaged over the five stocks, so a stock's trend never
counts as skill. It is judged against its **luck band** (§9). The verdict in §15 applies this rule literally:

- **"promising"** needs all six: the primary number is at least 2 luck-SDs above luck's centre (z ≥ 2); the
  direction skill is above 0; the model's per-name IC beats every obvious guess (drift, momentum, reversal); the leak
  detector is clean (§10); the per-name IC is above 0 in at least 4 of the 5 stocks; and it survives dropping the
  earnings weeks (z ≥ 1).
- **"no evidence of a large edge"**: z < 1, and the **95% band** around the primary number (its value ± 1.96
  luck-SDs: the true values this data cannot rule out) stays below 0.2. The verdict then says which small edge is
  still possible, and how large an edge this data would have found reliably.
- **"inconclusive"**: anything else. If the leak detector fires, the result is **void**.

A TEST read (§14) can only confirm a "promising" result (confirmed = a per-name IC above 0 with z ≥ 1.65 on TEST);
missing there means "not confirmed".

**Words you will meet**

| word | meaning |
|---|---|
| bar (hourly bar) | one hour of trading built from 60 one-minute bars; the day's last bar is 30 minutes (15:30–16:00) |
| session | one trading day, 09:30–16:00 New York (13:00 on a half-day) |
| decision time | when a forecast is made: at a bar's close (10:30, 11:30, …, 15:30, 16:00) |
| horizon | how far ahead: 35 hourly bars = 5 sessions = the same clock time next week |
| target / outcome | the 5-day change that really happened, from the decision's price to the price 5 sessions later |
| forecast | the model's guess of that change (the code calls it `edge`) |
| typical 5-day move | the stock's normal size of a 5-day move, from its last 20 sessions; the model works in these units so MU and CSCO are comparable |
| walk-forward | train on the past, forecast the next session, step forward, repeat: the way it would have run live |
| out of sample | a forecast about a moment after everything its model learned from |
| SD | standard deviation: the typical size of a number's random wobble |
| luck band | where a number lands 95% of the time in make-believe futures with no direction (the same forecasts; the obvious guesses rebuilt in each future): luck's centre ± 1.96 luck-SDs (§9) |
| z | how many luck-SDs a number sits from luck's centre; beyond ±2 happens by luck about 1 time in 20 |
| 95% band | the measured number ± 1.96 luck-SDs: the true values this data cannot rule out (as wide as the luck band, but centred on the measurement) |
| per-name IC | the rank correlation of forecast and outcome inside one stock, averaged over the stocks; negative = bigger forecasts went with smaller outcomes |
| independent weeks | how many separate answers the data holds: forecasts an hour apart share almost all of their 5-day outcome |

## 1 · The split: which months train, which are graded, which stay locked

**Question: which data is the model allowed to learn from, which forecasts are graded, and what stays untouched?**

The data is cut into three windows, the same in every darts_forecast notebook:

- **TUNE** (2025-06-18 → 2026-01-31): the only window this notebook reads. Settings may be chosen by looking at it.
- **TEST** (2026-02 → 2026-04): locked behind `READ_TEST = False` (§14). It is read once, with the settings frozen.
- **HOLDOUT** (2026-05 →): never read here.

Inside TUNE:

1. **Warm-up (21 sessions).** The inputs need history: the 20-day return and the volatility need 20 sessions.
   There is no model yet.
2. **History before the first fit (40 sessions).** The model waits until it has seen 40 sessions, so its first fit
   comes before the open of 2025-09-16.
3. **Graded.** From then on every hourly decision gets a forecast from a model that never saw its outcome.
4. **The last 5 sessions are graded nowhere.** Their 5-day outcome would end in TEST. Grading them would read TEST
   prices, and reading TEST later would then change TUNE numbers. So a forecast counts only when its decision AND
   its outcome are inside the window.

First, load the TUNE months: the stored 1-minute bars of the five stocks, adjusted for dividends (§2), with the
option inputs attached. Nothing after 2026-01-31 is loaded.

In [ ]:
panel = rs.load(settings, engine, adjust=True)  # TUNE only: 1-minute bars, dividend-adjusted
DATES = rs.split_dates(panel, P)  # the warm-up, the first fit, the last graded decision
print(f"{', '.join(panel.symbols)}: {len(DATES['sessions'])} sessions "
      f"{DATES['sessions'][0]} → {DATES['sessions'][-1]}, {len(panel.close):,} one-minute rows")

In [ ]:
display(rs.windows_table())

**The figure.** One horizontal bar per phase, on a calendar. **Grey**: warm-up (inputs only). **Green**: the history
the model waits for. **Blue**: the graded part, decisions from the first fit to the last decision whose 5-day outcome
still ends inside TUNE. **Orange, hatched**: the last 5 sessions, graded nowhere. **Hatched grey**: TEST and
HOLDOUT, locked.

**Look for:** how small the blue part is. Seven months of 1-minute data leave about 4 months of graded decisions
(90 sessions). §10 turns that into a number of independent outcomes.

In [ ]:
charts.split_timeline(windows=rs.WINDOWS, **DATES).show()

## 2 · The raw data: complete and clean?

**Question: is the 1-minute data complete and clean enough to build hourly bars from?**

The shared database stores one row per stock per minute of the regular session (09:30–16:00 New York). We check:

- **Completeness.** Every trading day of the stored daily bars must have its 390 one-minute bars, for every stock.
  A missing day would silently turn "5 sessions later" into 6. And every day needs its daily bar too: it carries the
  dividend adjustment (below).
- **Half-days.** On 2025-07-03, 2025-11-28 and 2025-12-24 the market closed at 13:00, but the database still holds
  bars after 13:00 (late prints). The code blanks them (`strategy.HALF_DAYS`).
- **Odd minutes.** Volume-0 minutes are copies of the previous price; they must be rare.
- **Splits.** A split would look like a huge overnight gap; none may be above 25%.
- **The window.** The data must end inside TUNE: TEST and HOLDOUT unread.
- **Option inputs.** They must cover the graded months.

First the hourly inputs are built (§3 explains them), because two checks count hourly bars.

In [ ]:
inp = rs.inputs(panel, P)  # every input on the hourly grid (strategy.build_inputs)
factors = rs.adjustment_factors(engine, panel.symbols, *rs.WINDOWS["tune"])  # daily bars' factor

In [ ]:
GRADED = (DATES["first_fit"], DATES["sessions"][-1])
cover = rs.coverage(panel.features, P["options"], *GRADED)  # share of name-days with option data
CHECKS += ck.data_checks(panel, inp, factors=factors, last_allowed=rs.WINDOWS["tune"][1],
                         coverage=cover)
display(rs.checks_table(CHECKS))

**Dividends.** The stored 1-minute bars are **not adjusted for dividends**. On an ex-dividend date (the first day a
buyer no longer gets the dividend) the price opens lower by about the dividend. A holder lost nothing: the dividend
is paid to them. But in the raw bars it looks like an overnight loss, and a 5-day change across that night would
be wrong by the dividend. `rs.load(..., adjust=True)` multiplies each day's 1-minute prices by the daily bars'
adjustment factor, which puts the dividend back.

**The table.** One row per ex-dividend date in TUNE: the dividend (%), the overnight gap in the raw bars (the fake
loss) and after the adjustment (what the model sees). **Look for:** CSCO's three dividends of about 0.6%. In the
raw bars each one is a −0.5 to −0.6% night; adjusted, the night is about flat.

One side effect: the adjustment scales every earlier price down a little (by the dividends paid since), so the
**prices shown in this notebook sit slightly below the traded ones** (CSCO by 1–2.6%, MU by about 0.2%, the other
three not at all). Every change in % is right.

In [ ]:
divs = rs.dividend_table(panel, factors)
display(divs.round(3))

**Earnings.** The biggest single moves of a stock come the morning after its quarterly report. The database's
`earnings` table gives the report dates (without the time of day: a report after the close moves the next morning,
one before the open moves the same day). They are not a model input; we use them in §10 to see whether a result
depends on a few report weeks.

In [ ]:
E = rs.earnings_dates(engine, panel.symbols, *rs.WINDOWS["tune"])  # report dates inside TUNE only
display(pd.Series({n: ", ".join(d) for n, d in E.items()}, name="report dates").to_frame())

## 3 · From 1-minute bars to hourly bars

**Question: what is one "hour" here, when are decisions made, and where does the night go?**

- **The bars.** Each session is cut into 7 bars starting at the 09:30 open: 09:30–10:29, 10:30–11:29, …,
  14:30–15:29 (60 minutes each) and a **last bar of 30 minutes**, 15:30–15:59.
- **The decision times.** A bar's close is the close of its last minute, and everything inside the bar is known
  then. A forecast is made at each close: **10:30, 11:30, 12:30, 13:30, 14:30, 15:30 and 16:00**.
- **"35 bars later"** is therefore always the same clock time 5 sessions later. On a half-day the three afternoon
  bars stay in the timeline, empty, so the count never shifts.
- **The night.** The 5-day change is measured close to close, so it includes the 5 overnight gaps (from one
  session's last price to the next session's first trade).
- **The 1-minute detail is summarized into inputs** (§5): the hour's return, its volatility, its range, buying vs
  selling volume, and more.

**The figure.** MU over six sessions around the worked example (7 to 14 November 2025). x = trading time, with
nights and the weekend squeezed out (each session has the same width; the ticks are the dates). **Grey line**: the
1-minute closes. **Blue dots**: the 7 hourly closes per day: the only prices the forecast is about. **Orange
segments**: the overnight gaps.

**Look for:** the orange jumps. Each night is one jump, often as big as a busy hour: the night into Monday 10
November (+4.2%) is bigger than every hour of these six sessions but one, and it happened while nobody could trade
and no hourly input changed.

In [ ]:
charts.hourly_day(panel, inp, NAME, "2025-11-07", days=6).show()

**How much of a move happens overnight?** Each day's close-to-close change splits into the overnight gap and the
trading day; over 5 sessions, into 5 gaps and 5 trading days. The table gives the nights' share of the variance
(how much of the "wobble" comes from the nights), the typical overnight gap and the typical 5-day move, in %.

**Look for:** the nights carry roughly a third or more of a 5-day move. That is why the target includes them
(`overnight = True`): without the nights the model would forecast a different, smaller quantity than the price
change you care about.

In [ ]:
display(rs.overnight_share(inp).round(1))

## 4 · The target: what number does the model try to predict?

**Question: for one decision, what exactly is "the 5-day change", and does the pipeline compute it correctly?**

- **The target** is the change from the price at the decision to the price 35 hourly bars later:
  log(price then ÷ price now). For small moves the log change is about the % change; log changes are used because
  they add up over time.
- **Typical-move units.** One model learns all five stocks, but CSCO typically moves ±3% in 5 days and COIN ±8%. So
  the target is divided by the stock's **typical 5-day move**, measured on its previous 20 sessions only. A target of
  −1 means "one typical 5-day move down". Every table below converts back to % and \$.
- **The check.** Recompute the change from the 1-minute bars as loaded (dividend-adjusted), for the worked example
  and 5 random decisions, and compare with the pipeline's number. The difference must be zero (below 1e-9).

**The table.** One row per decision: the decision and target times, both prices, the % change, the log change by
hand and from the pipeline, their difference, the typical 5-day move and the target in typical moves. The first
row is the worked example (MU, 2025-11-12, 13:30).

In [ ]:
EX = (rs.bucket_at(inp, TS), inp["names"].index(NAME))  # the example's bar and stock column
hand, row = ck.target_by_hand(panel, inp, n=5, include=[EX])
CHECKS.append(row)
display(hand.round(4))

**The worked example, in words.** At 13:30 on Wednesday 12 November 2025, MU stood at about \$246. Where would it be
at 13:30 the next Wednesday? The line below splits what happened into the nights and the trading hours.

In [ ]:
nt = rs.nights(inp, NAME, TS)
print(f"{NAME}: {rs.pct(nt['total'])} in log terms over 5 sessions; of that, the 5 nights "
      f"{rs.pct(nt['nights'])} ({', '.join(rs.pct(x) for x in nt['gaps'])}).")

**The figure: the question one forecast answers.** x = trading time (nights squeezed out), y = MU's price in \$.
**Black**: hourly closes. **Red dots and arrow**: the price at the decision and 5 sessions later, with the change.
**Shaded areas**: what the model can use at the decision: its daily summaries reach back 20 sessions (light green);
hourly, it reads its own 5-day change at the last 7 hourly closes, plus the last hour's inputs (blue); and what it
must predict, the 5 sessions ahead (orange).

**Look for:** the forecast is ONE number about the end of the orange area. Everything right of the decision is
unknown to the model.

In [ ]:
charts.target_picture(inp, NAME, EX[0]).show()

**The 5-day changes that will be graded** (decisions from the first fit to the last one with an outcome inside
TUNE), before any model: how often they were up, their mean and their spread, per stock.

**Look for the trend trap.** In these months MU went up in about 3 of every 4 weeks and COIN in about 1 of 3. A
forecaster that simply always says "up" for MU is right most of the time without knowing anything. Every direction
number in §9 is therefore compared with what such obvious guesses get, and judged inside each stock. (The "all" row
leaves the independent weeks blank: the stocks move together, so their weeks do not simply add up; §10 counts them.)

In [ ]:
t0 = rs.outcome_table(inp, P, "tune", earnings=E)  # every graded decision, before any model
display(rs.base_rates(t0).round(2))

**The figure.** One small panel per stock: the distribution of the graded 5-day changes (%). The solid line is 0,
the dotted line the mean; the title gives the share of up-weeks.

**Look for:** the widths differ about threefold between CSCO and COIN (hence the typical-move units), and MU's
mass sits right of 0, COIN's left of it.

In [ ]:
charts.outcome_hist(t0).show()

## 5 · What the model sees, and how far back

**Question: which information goes into one forecast, when is each piece known, and how long is the history
behind it?**

Seven groups of inputs, each known at the decision:

- **intraday** (from the 1-minute bars inside the last hour, known at its close): the hour's return, volatility,
  range, where it closed in its range, buying vs selling volume, volume against that hour's normal level, the
  return since the open, the distance to the day's average price;
- **market**: the other four stocks' average return in the last hour;
- **daily** (known at the previous close): yesterday's return, the last 5 and 20 sessions' returns, the 20-day
  volatility, where the price sits in its 20-day range, yesterday's close location and volume, and today's overnight
  gap (known at the open);
- **options** (known the evening before, never older than 5 days): implied volatility, skew, term structure,
  implied minus historical volatility, customers' put/call volume, option volume surge;
- **flags**: "this option source has data" (a missing value enters as 0, so the flag tells a real 0 from a gap);
- **calendar** (known in advance): clock time, weekday and option-expiry week of the target hour;
- **name**: which stock it is, so one model can learn all five.

**The table.** One row per input: its group, what it means, when it is known, and how many of its past values the
model reads ("lags").

In [ ]:
inputs = rs.input_table(inp, P)
print(f"numbers one forecast reads: {inputs.attrs['numbers']}")
display(inputs)

**The worked example's inputs.** The numbers the model read at 13:30 on 2025-11-12 for MU, each with its unit:
returns and ranges in typical moves (of an hour for the hourly inputs, of a day or 5 days for the daily ones),
positions inside a range on a fixed scale, volume and volatility levels as logs. A missing value would show as NaN
and enter as 0.

In [ ]:
display(rs.one_forecast_inputs(inp, NAME, TS).round(3))

### How long should the historic data input be? Three answers

"How much history" means three different things here; each is decided:

1. **The input window of one forecast: about one session in hourly detail, plus summaries reaching 20 sessions
   back.** The trees (`gbm`) see the latest hour of the hourly inputs (`cov_lags = 1`), their own target, the 5-day
   change, as it stood at each of the last 7 hourly closes (`lookback = 7`, one session of its path, which already
   covers about the last 6 sessions of returns), and the daily summaries: the 1-, 5- and 20-session returns,
   volatility and range. That is **44 numbers per forecast**.
   - *Why not longer:* each fit learns from only 7 to 25 independent weeks per stock (§6). Every extra column is
     another chance to fit noise: a week of hourly detail would be about 300 numbers. Nothing is known to make older
     hourly paths useful for 5-day direction (the earlier direction study of these five stocks,
     `strategies/burst_scan/DIRECTION_REPORT.md`, found nothing from 1 minute to 2 days).
   - *Why not shorter:* one session of the target's path is the least that shows "last week" in full.
   - *Not tuned:* the window is fixed by these reasons. Trying windows on TUNE and keeping the best would be another
     look. A neural network (TiDE, §12) reads its window as a sequence, so it gets one full horizon: 35 hours.
2. **The training history of each fit: all of it** (`expanding = True`). Each fit learns from every past decision
   whose 5-day outcome is already known, from the end of the warm-up on. At a 5-day horizon every fit is starved of
   independent outcomes, so throwing old weeks away (a rolling window) would be the costly mistake. The first fit
   waits for 40 sessions of history (`train_days = 40`): about 7 independent weeks per stock.
3. **The warm-up before anything: 21 sessions.** The 20-session inputs and the volatility scale need 20 sessions
   of 1-minute data first. (The stored daily bars could supply them and save those 21 sessions: a gap in §15.)

## 6 · Training: the walk-forward

**Question: when does the model learn, from which data, and is every graded forecast truly out of sample?**

- **Before every session** a fresh model is fitted (`retrain_minutes = 390`), on every past decision whose 5-day
  outcome is already known.
- **The 5 most recent sessions are skipped**, because their outcomes are still in the future. Their prices are
  known, but the price 5 sessions later is not yet.
- That model then forecasts every hour of the session. The next morning the next model takes over, having learned
  one more session of outcomes.
- So every graded forecast comes from a model that never saw its outcome, the way it would have run live. What is
  *not* out of sample is our choice of settings: TUNE is where they are chosen, and TEST exists to guard against
  that.

The walk-forward is computed once and saved on disk (about 2.5 minutes the first time).

In [ ]:
fc = rs.cached_forecast(panel, P)  # the walk-forward, or its saved copy (same knobs, same data)
log = fc["log"]
print(f"{len(log)} fits, {log['session'].iloc[0]} → {log['session'].iloc[-1]}, "
      f"{log['seconds'].sum() / 60:.1f} min of fitting")

**The figure.** One row per fit (fit 1 at the top), dates on x. **Blue**: the decisions it learns from (outcome
known). **Orange**: the 5 sessions it must skip (outcome still open). **Green**: the session it forecasts. The grey
area is the warm-up.

**Look for:** blue grows with every fit (expanding); orange always separates learning from forecasting; green never
overlaps blue.

In [ ]:
charts.fit_timeline(fc).show()

**The fit log** (every 10th fit): when the fit was made, which decisions it learned from, how many of them
(decisions × stocks with a known outcome), and how many **independent weeks per stock** that is. Neighbouring hourly decisions share almost all of their 5-day
outcome, so a stock adds one independent outcome per 5 sessions, not 35 rows.

**Look for:** the first fit learns from about 7 independent weeks per stock and the last one from about 25. That is
very little evidence for a model with 44 inputs.

In [ ]:
display(rs.fit_log(fc).iloc[::10])

**Checks: is it really out of sample?**

- *Bar by bar:* every forecast comes from the newest fit made at or before it; every fit learns only from outcomes
  known before it; training starts right after the warm-up; a fresh fit before every session; no forecast before
  the first fit.
- *The leak test:* replace every bar from 2026-01-16 at 10:30 onwards (10 sessions before the end) with fresh
  random bars and run the whole walk-forward again. Every input and every forecast made before that cut must stay
  exactly the same: if one changed, something reads the future. The forecasts after the cut must change, or the test
  could not fail. The cut sits one hour after a fit on purpose: that fit's first forecast (10:30) is made before the
  cut, so a fit that peeked at even one later outcome would change it. (The re-run is computed once and saved, about
  2.5 minutes the first time.)
- *The direct test:* fit one model on the data through 2026-01-15 alone (it physically lacks every later bar) and
  let it forecast 2026-01-16: it must give exactly the walk-forward's forecasts for that day. The latest forecast
  (§13) is made by the same code, so this also shows that it uses the walk-forward's own model.

In [ ]:
wf = ck.walk_forward_checks(fc, P) + ck.truncation_check(panel, P, fc, rs.cached_forecast)
DAY = DATES["sessions"][-11]  # the direct test fits on the data through this session only
wf += ck.refit_check(fc, *rs.refit_forecast(panel, P, DAY, fc["inputs"]), DAY)
CHECKS += wf
display(rs.checks_table(wf))

## 7 · One forecast, end to end

**Question: what did the model say for MU at 13:30 on 2025-11-12, and what happened?**

From here on every graded forecast is one row of a table: the decision time, the price at that moment, the
forecast, the price 5 sessions later and what the obvious guesses would have said. The forecast (in typical moves)
times the typical 5-day move is the forecast as a log change; e^(that) − 1 is the forecast in %, and today's price ×
e^(that) is the **predicted price**.

In [ ]:
t = rs.add_baselines(rs.horizon_table(fc, "tune", earnings=E), inp)  # one row per graded forecast
CHECKS += ck.window_checks(t, rs.WINDOWS["tune"], expected=t0)
print(f"{len(t):,} graded forecasts: {t['ts'].min():%Y-%m-%d} → {t['ts'].max():%Y-%m-%d}")

**The worked example.** The forecast in % and as a price, next to the actual price and the error. One case proves
nothing: it only shows what each number means. §8 and §9 look at all of them.

In [ ]:
display(rs.one_forecast(t, NAME, TS).to_frame())

## 8 · Predicted price vs actual price

**Question: does the predicted price track the real one, and does that mean the forecast is good?**

**The trap first.** The predicted price in 5 days = today's price × (1 + the predicted change). The change is a few
%, but over months the price itself wanders by tens of % (MU more than doubled in these months). So ANY forecast
that starts from today's price looks like it follows the real price: even "no change" (simply today's price) does.

**The figure (the flattering view).** x = the target time (each prediction drawn at the time it is about, nights
squeezed out), y = MU's price in \$. **Black**: the actual price. **Blue**: the model's predicted price, made 5
sessions earlier. **Dotted grey**: "no change", the price 5 sessions earlier.

**Look for:** blue and grey are close to each other and both seem to "track" black, about a week late. The grey line
knows nothing about direction, so this picture cannot show skill.

In [ ]:
charts.price_vs_forecast(t, NAME).show()

**The numbers behind the picture**, per stock (and all together where it makes sense):

- **R² price**: how much of the price's ups and downs over the months the prediction "explains", for the model and
  for "no change". **Look for:** "no change" scores high (0.3 to 0.9) although it forecasts nothing: that is the
  flattery. The model's R² is lower in every stock but CSCO: its nudges make even the price level worse.
- **error %**: the typical price error in % of the actual price, for both. **Theil U** = model error ÷ "no change"
  error: **below 1 means better than not forecasting at all**, above 1 worse.
- **R² change**: the same question asked about the 5-day change, the honest version. Above 0 = better than
  forecasting "no change"; below 0 = worse.

In [ ]:
prices = rs.price_table(t)
display(prices.round(3))

**What does the predicted price really track?** For each prediction, compare it with the actual price k sessions
before its target time, k = 0 … 10. k = 0 is the price it is about; k = 5 is the price when the forecast was made.

**The figure.** x = k, y = the typical gap between the predicted price and that actual price (%). **Look for:** the
gap is smallest at k = 5: the predicted price is mostly "today's price, nudged". The nudge (the forecast itself) is
the only part that could carry skill.

In [ ]:
charts.lag_profile(rs.lag_profile(t, NAME)).show()

**The honest view.** x = the predicted 5-day change (%), y = the actual 5-day change (%), every graded forecast,
one colour per stock. The dashed diagonal is a perfect forecast; the grey lines split "called up / called down" and
"went up / went down".

**Look for:** a forecast with skill tilts the cloud up along the diagonal. A round cloud means the change is not
predicted. Vertical streaks are runs of hourly forecasts that barely change while the outcome moves (neighbouring
forecasts share most of their 5-day window).

In [ ]:
charts.return_scatter(t).show()

**How big are the forecasts?** A model that knows little should predict small changes, close to its average: its
price error is then about that of "no change". A model that predicts big changes without getting their direction
makes a bigger error than "no change", which is what a Theil U above 1 in the table says.

In [ ]:
print(f"the forecasts vary {t['f_pct'].std() / t['r_pct'].std():.0%} as much as the actual changes "
      f"(SD {t['f_pct'].std():.1%} vs {t['r_pct'].std():.1%})")

## 9 · Direction: does it call up vs down better than chance and the obvious guesses?

**Question: how often is the direction right, compared with luck and with the obvious guesses?**

- **Hit rate:** how often the sign of the forecast matches the sign of the outcome.
- **The base-rate trap:** in a stock that went up in 3 of 4 weeks, "always up" hits 75% while knowing nothing.
- **Chance:** the hit rate of a coin that says "up" as often as the model does, in a market that went up as often
  as this one: chance = q × b + (1 − q) × (1 − b), with q = how often the forecast says up and b = how often the
  stock went up. **Direction skill = hit − chance**, computed per stock and averaged: 0 = no skill, whatever the
  trend.
- **Per-name IC** (the primary number, §0) also uses the size of the forecast: are the big up-calls followed by the
  big up-moves?
- **The obvious guesses**, made with only what was known at each decision, on exactly the same rows: **always up**;
  **drift** (the stock's average 5-day change so far); **momentum** (the last 5 days continue); **reversal** (they
  reverse). The **classifier** is the same trees trained only on up vs down; it gives P(up), joined here as
  "up if P(up) > 50%".

**The luck band.** Keep every forecast as it is. Build 1,000 make-believe futures in which each real hourly move
keeps its size but gets a random direction (one coin per hour for all five stocks, because they move together; the
nights are part of the first hour of each session). Nothing can be predicted in those futures, so the spread of a
score against them is **what luck alone gives**. A number counts as "more than luck" when it stands about 2 of
luck's SDs (z ≥ 2) from luck's centre. The usual textbook error bars treat each day as independent and raise false
alarms about 1 time in 4 at this horizon; this band about 1 time in 40 (both measured on coin-flip copies of these
data while this notebook was designed). The obvious guesses made from the price path itself (drift, momentum,
reversal) are rebuilt inside every make-believe future, so their bands sit where luck puts *them* (see the cautions
below); the model's and the classifier's forecasts are kept as they are.

In [ ]:
flips = rs.flip_outcomes(inp, t, draws=DRAWS)  # 1,000 make-believe outcomes for every graded row
fc_clf = rs.cached_forecast(panel, {**P, "model": "gbm_clf"})  # the up/down classifier, P(up)

**The table.** One row per forecast: how often it says up, its hit rate, chance, the skill (points) and its z,
the per-name IC, where luck centres it and its z, and the pooled IC (all stocks at once: shown, but it rewards
knowing which stock trended, so it is never used). The last two rows compare the model with drift and momentum on
the same futures (paired).

In [ ]:
t = rs.add_p_up(t, fc_clf)
sc = rs.scores(t, flips)  # every metric with its luck band
display(rs.score_table(sc).round(2))

**The figure: the estimate-against-luck chart** (it is used for every comparison from here on). One row per
forecast. The **grey bar** is the luck band, where luck alone puts the number 95% of the time; the **dot** is the
measured value (blue: the model; purple: the classifier; black: the obvious guesses). A dot outside its grey bar is
more than luck. A dot outside it on the negative side is not proof that the forecast knows the opposite, and with
several rows a |z| near 2 is borderline: one in twenty rows lands there by luck.

First the **direction skill** in points (hit − chance):

In [ ]:
charts.dots(sc, "skill", title="Direction skill (points): measured vs where luck lands").show()

Then the **per-name IC**, the primary number.

**Two cautions for the obvious guesses.** Momentum and reversal are mirror images, so one always looks good when
the other looks bad. And a guess built from the recent path (drift, momentum) scores *below* zero within a short
sample even when nothing is predictable: the guess and the outcomes that follow are pieces of the same price path,
and inside 18 weeks they must even out. The luck band rebuilds these guesses inside every make-believe future, so it
contains that effect: their grey bars are centred away from 0 (far left for drift; the "IC luck centre" column of
the table above gives each centre). Judge every dot against its own bar, never against 0.

In [ ]:
charts.dots(sc, "ic_name", scale=1, unit="",
            title="Per-name IC: measured vs where luck lands").show()

**Is it more right when it is more sure?** Each stock's forecasts are split into thirds by their size: weak, medium,
strong. A model that knows something is right more often on its strong calls, the ones a trader would act on.

**The figure.** One row per third, the direction skill (points) against its luck band. **Look for:** a staircase
rising from weak to strong. Flat or jumpy dots mean the size of the forecast carries no information.

In [ ]:
conf = rs.scores(t, flips, cols=("edge",), by="confidence", pairs=())
charts.dots(conf, "skill", rows="group", where={"forecast": "edge"},
            title="Direction skill (points) of the model's weak, medium and strong calls").show()

**Is the classifier's probability honest?** When it says "60% chance of up", does the price go up about 60% of the
time? The forecasts are split into fifths by their P(up).

**The figure.** x = the P(up) a fifth of the forecasts got (its mean), y = the share of them that really went up,
with an error bar (the measured share ± 1.96 luck-SDs): an honest probability's diagonal passes through the bars.
**Look for:** dots along the diagonal AND spread out. A flat row of dots means the probabilities carry no
information; dots that fall as P(up) rises mean they point the wrong way.

The **Brier score** (the mean of (P(up) − outcome)², outcome 1 for up and 0 for down; lower is better) is compared
with always saying 50% (0.250) and with the fair reference: each stock's own past share of up-weeks. (It gets no luck
band: half the make-believe weeks go up, so there any spread-out P(up) would lose to the past up share anyway.)

In [ ]:
rel = rs.reliability(t, flips=flips)
charts.reliability(rel).show()
a = rel.attrs
print(f"Brier score {a['brier']:.3f}; always 50%: {a['brier_coin']:.3f}; each stock's past up "
      f"share: {a['brier_trailing']:.3f}. Brier improvement over the past up share: "
      f"{a['skill_vs_trailing']:+.0%}; P(up) spread (SD) {a['sharpness']:.2f}")

## 10 · How much evidence: what could luck do, what could we see, is the pipeline honest?

**Question: how many independent outcomes do 3,090 graded forecasts hold, how large an edge could they reveal, and
does the pipeline invent skill where none can exist?**

- **Overlap.** Two forecasts an hour apart look at 34 of the same 35 future hours: almost the same bet. A stock
  adds one independent outcome per 5 sessions.
- **Co-movement.** The five stocks move together, so they count as fewer than five.

**The ledger.** Graded rows, sessions, independent weeks per stock, the stocks' mean correlation, how many stocks
they are "worth", and the independent outcomes in all.

In [ ]:
ledger = rs.effective_n(t)
display(ledger.round(2))

**How large an edge this data would find.** An edge is found 4 times in 5 (at the 5% level) only if it is about 2.8
luck-SDs large. This is a before-the-data number: it says how sensitive the test is, not what the data showed (what
the data rule out is the 95% band, in §15's verdict). In plain words:

In [ ]:
prim, skill = rs.pick(sc), rs.pick(sc, "edge", "skill")  # the model's IC and skill with their bands
small = rs.found_chance(0.10, prim["null_sd"])  # how often an IC of 0.10 would be found
print(f"with 7 months of hourly data an edge of a per-name IC of about "
      f"{2.8 * prim['null_sd']:.2f} (a direction skill of about {280 * skill['null_sd']:.0f} "
      f"points) would be found 4 times in 5; a smaller one less often: an IC of 0.10 only "
      f"{small:.0%} of the time")

**Is it spread out, or one stock or a few report weeks?** The per-name IC of each stock on its own (each with its
own, much wider luck band), and the primary number without the forecasts whose 5-day window holds an earnings
report. These breakdowns can only veto a result (an "edge" that is all one report week); they can never create one.

In [ ]:
by_name = rs.scores(t, flips, cols=("edge",), by="name", pairs=())
by_earn = rs.scores(t, flips, cols=("edge",), by="earn", pairs=())
charts.dots(by_name, "ic_name", rows="group", where={"forecast": "edge"}, scale=1, unit="",
            title="Per-name IC of the model, stock by stock").show()

In [ ]:
calm = rs.pick(by_earn, "edge", "ic_name", False)
print(f"without the earnings windows ({t['earn'].mean():.0%} of the rows): per-name IC "
      f"{calm['value']:+.3f}, z {calm['z']:+.1f}")

**The leak detector.** The luck band keeps the model's forecasts fixed, so it cannot catch a pipeline that peeks at
the future. Here the **whole pipeline** (inputs, every daily fit, every forecast) runs again on make-believe markets in
which each 1-minute move got a random direction (one coin per minute for all five stocks; volatility, volume, the
calendar and the option inputs stay real). There is nothing to predict in them, so each must land inside its own
luck band. A clearly positive average would mean something reads the future, and the result would be void. (Three
markets, about 2.5 minutes each the first time, then saved.)

In [ ]:
null = rs.null_scores(panel, P, seeds=NULL_SEEDS)
CHECKS.append(("leak detector: the coin-flip markets find nothing beyond luck", null.attrs["clean"],
               f"combined z {null.attrs['combined_z']:+.2f} (a leak shows as z >= 2)"))
display(rs.null_table(null).round(3))

**The figure.** One row per coin-flip market and one for the real market: the per-name IC (dot) against its own luck
band (grey bar). **Look for:** the coin-flip dots inside their bars (no leak), and whether the real dot stands out
from them.

In [ ]:
charts.dots(rs.null_rows(null, sc), "ic_name", scale=1, unit="",
            title="Per-name IC on coin-flip markets (nothing to find) and on the real one").show()

## 11 · Does retraining help?

**Question: does a model refitted before every session forecast better than one trained once, or weekly?**

The same forecaster on exactly the same graded rows and the same make-believe futures, refitted on three schedules:

- **once**: trained before 2025-09-16 and never again (the control);
- **weekly**: refitted every 5 sessions;
- **every session**: refitted before every session (this notebook's setting).

The "… − once" rows are **paired** differences: same rows, same outcomes, only the model's freshness differs.

**The figures.** Per-name IC, then direction skill, one row per schedule and per difference, against luck.
**Look for:** a schedule clearly better than "once". With several comparisons, one |z| near 2 can be luck.

In [ ]:
cad = rs.cadence_scores(panel, P, t, flips, cadences=(0, 1950, 390))  # each walk-forward cached
print({k: f"{n} fit{'s' * (n != 1)}, {s / 60:.1f} min" for k, (n, s) in cad.attrs["fits"].items()})
charts.dots(cad, "ic_name", scale=1, unit="", title="Per-name IC by retraining schedule").show()

In [ ]:
charts.dots(cad, "skill", title="Direction skill (points) by retraining schedule").show()

**The same as numbers, then the answer in words.** If a schedule is clearly *worse* than training once, the likely
reason is that each refit chases the last few weeks: with only 7 to 25 independent weeks per stock, one new week
moves the model a lot, and the next week rarely repeats it.

In [ ]:
display(rs.score_table(cad)[["skill (pts)", "skill z", "per-name IC", "per-name IC z"]].round(2))
print(rs.cadence_words(cad))

## 12 · Other models (optional: `RUN_DEEP`)

**Question: does a neural network (TiDE) do better, and are its 80% ranges honest?**

TiDE reads the last 35 hours of every input as a sequence and predicts five quantiles of the 5-day change. Its 10%
and 90% quantiles bracket an **80% range**: an honest range holds the actual price 8 times in 10, with 1 miss below
and 1 above. The table gives the share inside, below and above, and the typical width, for all forecasts and split
by earnings windows. With about 50 independent outcomes, anything from about 70% to 90% inside is consistent with
an honest 80% range.

It is off by default (`RUN_DEEP = False`): a daily-refitted TiDE takes several minutes, and it is a second model, not
a second chance for the verdict.

In [ ]:
DEEP = {"model": "tide", "lookback": 35, "val_days": 0, "n_epochs": 10}
if RUN_DEEP:
    t_deep = rs.horizon_table(rs.cached_forecast(panel, {**P, **DEEP}), "tune", earnings=E)
    display(rs.coverage_table(t_deep).round(1))
else:
    print("skipped (RUN_DEEP = False)")

## 13 · The latest forecast: the end of the pipeline

**Question: what does the model say now about the next 5 days?**

The walk-forward only makes forecasts whose outcome is already in the data, so the last 5 sessions never get one.
A live run needs one more step: fit one fresh model on every outcome known at the last bar (the same expanding
window), and forecast from the last bar: Friday 2026-01-30 at 16:00, about Friday 2026-02-06 at 16:00. No market
holiday falls in between, which matters: past the data's end the code's calendar knows no holidays (§15). That
target lies in the locked TEST window, so we do not look at what happened.

**The table.** Per stock: the last price, the forecast change (%), the predicted price, the classifier's P(up),
and the stock's typical range of 5-day changes (its 10% and 90% points over TUNE), for scale.

In [ ]:
latest = rs.latest_forecast(panel, P, clf=True)  # a fresh fit on every known outcome, then predict
display(latest.round(3))

**The figure.** x = trading time: MU's last 20 sessions and the 5 ahead; y = price in \$. **Black**: the hourly
closes. **Grey cone**: the typical range of a 5-day change (10% to 90%) around today's price: the stock's own history
in TUNE, not a forecast. It leans up for MU because MU rose in 3 of 4 TUNE weeks. **Dotted**: "no change". **Blue
dot**: the predicted price.

**Look for:** where the dot sits inside the cone. Given everything in §9–§11, read it as the model's guess, not as
information.

In [ ]:
charts.fan(latest, inp, NAME).show()

## 14 · TEST, once (locked: `READ_TEST = False`)

**Question: does the frozen setting hold on three months it has never seen?**

Run this only after the settings of §0 are final, and only once. The panel is loaded through 2026-04-30 (never
later; HOLDOUT stays unread) and the walk-forward simply continues. A TEST forecast counts when its outcome also ends
inside TEST: about 11 independent weeks per stock. So TEST can catch a collapse or a flipped sign, and can confirm a
"promising" TUNE result; it cannot confirm a small edge.

In [ ]:
sc_test = None
if READ_TEST:
    panel_t = rs.load(settings, engine, end=rs.WINDOWS["test"][1], adjust=True)
    fc_t = rs.cached_forecast(panel_t, P)
    e_t = rs.earnings_dates(engine, panel.symbols, *rs.WINDOWS["test"])
    t_t = rs.add_baselines(rs.horizon_table(fc_t, "test", earnings=e_t), fc_t["inputs"])
    sc_test = rs.scores(t_t, rs.flip_outcomes(fc_t["inputs"], t_t, draws=DRAWS))
    display(rs.score_table(sc_test).round(2))
else:
    print("TEST not read (READ_TEST = False)")

## 15 · Gaps, checks and the verdict

**Question: what is missing, did every step do what it claims, and what is the answer?**

**The gaps** found along the way, with what this run measured for each and the fix. "data" gaps limit what any
model can learn here; "code" gaps are things the pipeline does not do yet; "method" gaps are traps in judging a
5-day forecast made every hour.

In [ ]:
gaps = rs.gaps_table(rs.gap_numbers(t, sc, ledger, divs, prices, panel))
display(gaps)

**Every check**, from §2, §4, §6, §7 and §10. All must pass; a failed check makes the numbers above untrustworthy.

In [ ]:
ALL = rs.checks_table(CHECKS)
display(ALL)
print(f"{int(ALL['ok'].sum())} of {len(ALL)} checks pass")

**The verdict**, the rule of §0 applied literally: first each of its tests, then the answer in three sentences.

In [ ]:
v = rs.verdict(sc, null, {"name": by_name, "earnings": by_earn}, test=sc_test)
display(pd.Series(v["tests"], name="passed").to_frame())
print(rs.verdict_words(v, sc, ledger))

One machine-readable line records the run, like every notebook of this repo:

In [ ]:
rs.scorecard(rs.walk_summary(P, t, sc, ledger, null, cad, latest, v, ALL))

**What would make the question answerable.** Every gap above comes back to one number: about 18 independent weeks
per stock. Three ways to get more, none needing a new data source:

- **more stocks:** about 510 symbols have 1-minute bars over TUNE (no backfill). Stocks move together, though, so
  by §10's rule 50 stocks count as far fewer than 50 independent ones: the line below works it out;
- **daily bars:** stored since 2015, about 25 times the evidence for a 5-day question (the hourly detail could then
  be tested as an add-on);
- **a shorter horizon:** a 1-day forecast gives 5 times as many independent outcomes.

Trading a 5-day forecast needs a rule that holds overnight (`valid()` rejects `overnight` today): a later step, worth
taking only once a forecast earns it.

In [ ]:
rho = ledger.loc["mean correlation between the names", "value"]
now, more = ledger.loc["effective names", "value"], 50 / (1 + 49 * rho)  # §10's rule, 50 stocks
print(f"50 stocks that move together like these five count as about {more:.1f} independent "
      f"stocks ({now:.1f} now): {more / now:.1f} times the evidence, so the edge found reliably "
      f"would fall from an IC of about {v['detectable']:.2f} to about "
      f"{v['detectable'] * (now / more) ** 0.5:.2f}; more only if the added stocks move less "
      f"together")